# MGCS — Exploratory Analysis

This notebook walks through:
1. Loading a small HaluEval sample
2. Computing MGCS scores at all three levels
3. Visualising score distributions for faithful vs. hallucinated texts
4. Plotting the ROC curve for S_final vs. single-level baselines

In [ ]:
import sys
sys.path.insert(0, '..')   # run from notebooks/ directory

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

from src.aggregation.aggregator import MGCSAggregator
from src.evaluation.halueval_loader import load_halueval
from src.evaluation.metrics import compute_metrics

## 1. Load a small HaluEval QA sample

In [ ]:
records = load_halueval(split_name='qa', max_samples=100)
print(f"Loaded {len(records)} records")
print("Example:", records[0])

## 2. Compute MGCS scores

In [ ]:
aggregator = MGCSAggregator(alpha=0.3, beta=0.4, gamma=0.3)

rows = []
for rec in records:
    scores = aggregator.compute(rec['generated'], rec['reference'])
    rows.append({
        'label': rec['label'],
        's_word': scores.s_word,
        's_sent': scores.s_sent,
        's_doc': scores.s_doc,
        's_final': scores.s_final,
    })

df = pd.DataFrame(rows)
df['label_name'] = df['label'].map({0: 'Faithful', 1: 'Hallucinated'})
df.head()

## 3. Score distributions per label

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
cols = ['s_word', 's_sent', 's_doc', 's_final']
titles = ['S_word (α=0.3)', 'S_sent (β=0.4)', 'S_doc (γ=0.3)', 'S_final']

for ax, col, title in zip(axes, cols, titles):
    for label, grp in df.groupby('label_name'):
        ax.hist(grp[col], bins=20, alpha=0.6, label=label)
    ax.set_title(title)
    ax.set_xlabel('Cosine Similarity')
    ax.legend()

plt.suptitle('Score Distributions: Faithful vs Hallucinated', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('../results/score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. ROC Curves — S_final vs. single-level baselines

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for col, label in [('s_word','S_word'), ('s_sent','S_sent'), ('s_doc','S_doc'), ('s_final','S_final (MGCS)')]:
    # Invert: low similarity → likely hallucinated
    inv = 1 - df[col]
    fpr, tpr, _ = roc_curve(df['label'], inv)
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if col == 's_final' else 1.5
    ax.plot(fpr, tpr, lw=lw, label=f'{label} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Hallucination Detection')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Weight sensitivity analysis (grid search over α, β, γ)

In [ ]:
from src.evaluation.metrics import compute_metrics
import itertools

results = []
step = 0.1
candidates = [round(i * step, 1) for i in range(11)]

for a, b in itertools.product(candidates, candidates):
    g = round(1.0 - a - b, 1)
    if g < 0 or g > 1:
        continue
    s_final = a * df['s_word'] + b * df['s_sent'] + g * df['s_doc']
    m = compute_metrics(s_final.tolist(), df['label'].tolist())
    results.append({'alpha': a, 'beta': b, 'gamma': g, **m})

res_df = pd.DataFrame(results).sort_values('AUROC', ascending=False)
print("Top 5 weight combinations by AUROC:")
print(res_df.head())